# 3A. Reporte escrito. Experimentos y análisis de algoritmos de ordenamiento.

Jovanna del Rocío Aguilar López

### Introducción

Los algoritmos de ordenamiento son una parte fundamental en el análisis y procesamiento de datos, ya que permiten organizar la información de manera eficiente para su posterior uso. En este trabajo se implementaron cuatro algoritmos de ordenamiento: HeapSort, MergeSort, QuickSort utilizando la mediana como pivote y Bubble sort adaptativo. El objetivo fue comparar su desempeño utilizando distintos conjuntos de datos que presentan diferentes niveles de perturbación.

Para evaluar el comportamiento de los algoritmos se consideraron dos medidas principales: el número de comparaciones realizadas y el tiempo de ejecución necesario para ordenar los datos. A partir de estos resultados se elaboraron tablas y gráficas que permiten observar con mayor claridad las diferencias entre los métodos.

De esta manera, el análisis permite identificar patrones en el desempeño de los algoritmos frente a distintos niveles de desorden en los datos y comparar qué tan eficientes resultan en cada caso.

In [50]:
function heapify_count!(A, n, i, comparaciones)
    # Al inicio suponemos que el nodo actual es el mayor
    mayor = i

    # Calculamos las posiciones de los hijos
    izquierdo = 2 * i
    derecho = 2 * i + 1

    # Si existe el hijo izquierdo, se compara con el mayor actual
    if izquierdo <= n
        comparaciones[] += 1
        if A[izquierdo] > A[mayor]
            mayor = izquierdo
        end
    end

    # Si existe el hijo derecho, se compara con el mayor actual
    if derecho <= n
        comparaciones[] += 1
        if A[derecho] > A[mayor]
            mayor = derecho
        end
    end

    # Si alguno de los hijos es mayor, se intercambian
    if mayor != i
        A[i], A[mayor] = A[mayor], A[i]

        # Se sigue ajustando el subárbol afectado
        heapify_count!(A, n, mayor, comparaciones)
    end
end

function heapsort_count!(A)
    # Tamaño del arreglo
    n = length(A)

    # Contador de comparaciones
    comparaciones = Ref(0)

    # Primero se construye el max-heap
    for i in fld(n, 2):-1:1
        heapify_count!(A, n, i, comparaciones)
    end

    # Después se va sacando el máximo y se reconstruye el heap
    for i in n:-1:2
        A[1], A[i] = A[i], A[1]
        heapify_count!(A, i - 1, 1, comparaciones)
    end

    return A, comparaciones[]
end

heapsort_count! (generic function with 1 method)

**MergeSort** es un algoritmo de ordenamiento basado en la estrategia de divide y vencerás. El arreglo se divide recursivamente en dos mitades hasta obtener subarreglos de tamaño uno, que ya se consideran ordenados. Después, estos subarreglos se combinan mediante un proceso de mezcla que compara los elementos de ambas partes y los coloca en el orden correcto. Este procedimiento se repite hasta reconstruir el arreglo completo. MergeSort tiene una complejidad temporal de $(O(n \log n))$ en todos los casos, lo que lo hace un algoritmo estable y con un desempeño predecible. *(Knuth, 1998; Cormen et al., 2022, cap. 2).*

In [53]:
function mergesort_count(A)
    # Se hace una copia para no modificar el original
    B = copy(A)

    # Contador de comparaciones
    comparaciones = Ref(0)

    # Se ordena todo el arreglo
    mergesort_recursivo!(B, 1, length(B), comparaciones)

    return B, comparaciones[]
end

function mergesort_recursivo!(A, izquierda, derecha, comparaciones)
    # Si solo hay un elemento, ya está ordenado
    if izquierda >= derecha
        return
    end

    # Se calcula la mitad
    mitad = (izquierda + derecha) ÷ 2

    # Se ordena la mitad izquierda
    mergesort_recursivo!(A, izquierda, mitad, comparaciones)

    # Se ordena la mitad derecha
    mergesort_recursivo!(A, mitad + 1, derecha, comparaciones)

    # Se juntan ambas mitades
    merge_count!(A, izquierda, mitad, derecha, comparaciones)
end

function merge_count!(A, izquierda, mitad, derecha, comparaciones)
    # Copias de las dos mitades
    I = A[izquierda:mitad]
    D = A[mitad + 1:derecha]

    # Índices para recorrer ambas mitades
    i = 1
    j = 1
    k = izquierda

    # Se compara el primer elemento disponible de cada mitad
    while i <= length(I) && j <= length(D)
        comparaciones[] += 1
        if I[i] <= D[j]
            A[k] = I[i]
            i += 1
        else
            A[k] = D[j]
            j += 1
        end
        k += 1
    end

    # Se copian los elementos que queden en la izquierda
    while i <= length(I)
        A[k] = I[i]
        i += 1
        k += 1
    end

    # Se copian los elementos que queden en la derecha
    while j <= length(D)
        A[k] = D[j]
        j += 1
        k += 1
    end
end

merge_count! (generic function with 1 method)

**QuickSort** es un algoritmo de ordenamiento que funciona seleccionando un elemento del arreglo como pivote y reorganizando los demás elementos alrededor de él. Los elementos menores o iguales al pivote se colocan a la izquierda y los mayores a la derecha. Posteriormente, el mismo procedimiento se aplica recursivamente a cada una de las particiones hasta que todo el arreglo queda ordenado. En promedio, su complejidad es \(O(n \log n)\), aunque en el peor caso puede llegar a \(O(n^2)\). *(Knuth, 1998; Cormen et al., 2022).*

En esta implementación el pivote se selecciona utilizando la estrategia de **mediana de tres**, que consiste en elegir como pivote la mediana entre el primer elemento, el elemento central y el último del subarreglo. Esta técnica se utiliza para reducir la probabilidad de particiones muy desbalanceadas y mejorar el desempeño del algoritmo en la práctica.

In [56]:
function mediana_de_tres!(A, inicio, fin, comparaciones)
    # Se calcula la posición central
    centro = (inicio + fin) ÷ 2

    # Se ordenan los tres candidatos
    comparaciones[] += 1
    if A[inicio] > A[centro]
        A[inicio], A[centro] = A[centro], A[inicio]
    end

    comparaciones[] += 1
    if A[inicio] > A[fin]
        A[inicio], A[fin] = A[fin], A[inicio]
    end

    comparaciones[] += 1
    if A[centro] > A[fin]
        A[centro], A[fin] = A[fin], A[centro]
    end

    # La mediana queda en el centro
    return centro
end

function particion_mediana!(A, inicio, fin, comparaciones)
    # Se obtiene el pivote con mediana de tres
    indice_pivote = mediana_de_tres!(A, inicio, fin, comparaciones)

    # El pivote se manda al final
    A[indice_pivote], A[fin] = A[fin], A[indice_pivote]
    pivote = A[fin]

    # i marca la posición de los menores o iguales al pivote
    i = inicio - 1

    for j in inicio:fin-1
        comparaciones[] += 1
        if A[j] <= pivote
            i += 1
            A[i], A[j] = A[j], A[i]
        end
    end

    # El pivote se coloca en su lugar final
    A[i + 1], A[fin] = A[fin], A[i + 1]

    return i + 1
end

function quicksort_mediana_recursivo!(A, inicio, fin, comparaciones)
    # Solo se sigue si hay más de un elemento
    if inicio < fin
        posicion_pivote = particion_mediana!(A, inicio, fin, comparaciones)

        # Se ordena la parte izquierda
        quicksort_mediana_recursivo!(A, inicio, posicion_pivote - 1, comparaciones)

        # Se ordena la parte derecha
        quicksort_mediana_recursivo!(A, posicion_pivote + 1, fin, comparaciones)
    end
end

function quicksort_mediana_count!(A)
    # Contador de comparaciones
    comparaciones = Ref(0)

    # Se llama al algoritmo sobre todo el arreglo
    quicksort_mediana_recursivo!(A, 1, length(A), comparaciones)

    return A, comparaciones[]
end

quicksort_mediana_count! (generic function with 1 method)

**Bubble Sort adaptativo** es una variante del algoritmo clásico de bubble sort, en el cual se comparan elementos adyacentes y se intercambian cuando están fuera de orden. En la versión adaptativa se introduce una condición que permite detener el algoritmo si durante una pasada completa no se realizan intercambios, lo que indica que el arreglo ya está ordenado. Esta característica permite aprovechar instancias casi ordenadas, aunque en el peor caso su complejidad sigue siendo $(O(n^2))$. Este tipo de comportamiento se relaciona con los algoritmos adaptativos estudiados en la literatura de ordenamiento. *(Estivill-Castro y Wood, 1992; Cormen et al., 2022).*

In [59]:
function bubble_sort_adaptativo_count!(A)
    # Tamaño del arreglo
    n = length(A)

    # Contador de comparaciones
    comparaciones = 0

    # En cada pasada el mayor va quedando al final
    for i in 1:n-1
        # Sirve para ver si hubo cambios en la pasada
        intercambios = 0

        # Se comparan elementos vecinos
        for j in 1:n-i
            comparaciones += 1
            if A[j] > A[j+1]
                A[j], A[j+1] = A[j+1], A[j]
                intercambios += 1
            end
        end

        # Si no hubo intercambios, ya está ordenado
        if intercambios == 0
            break
        end
    end

    return A, comparaciones
end

bubble_sort_adaptativo_count! (generic function with 1 method)

In [61]:
using JSON3

# Ruta de la carpeta donde están los archivos
ruta_carpeta = "/Users/enriquear/Documents/listas-posteo-con-perturbaciones"

archivo_p016 = "listas-posteo-con-perturbaciones-p=016.json"
ruta_p016 = joinpath(ruta_carpeta, archivo_p016)
contenido_p016 = JSON3.read(read(ruta_p016, String))
datos_p016 = Int.(contenido_p016["reunion"])


archivo_p032 = "listas-posteo-con-perturbaciones-p=032.json"
ruta_p032 = joinpath(ruta_carpeta, archivo_p032)
contenido_p032 = JSON3.read(read(ruta_p032, String))
datos_p032 = Int.(contenido_p032["reunion"])


archivo_p064 = "listas-posteo-con-perturbaciones-p=064.json"
ruta_p064 = joinpath(ruta_carpeta, archivo_p064)
contenido_p064 = JSON3.read(read(ruta_p064, String))
datos_p064 = Int.(contenido_p064["reunion"])


archivo_p128 = "listas-posteo-con-perturbaciones-p=128.json"
ruta_p128 = joinpath(ruta_carpeta, archivo_p128)
contenido_p128 = JSON3.read(read(ruta_p128, String))
datos_p128 = Int.(contenido_p128["reunion"])


archivo_p256 = "listas-posteo-con-perturbaciones-p=256.json"
ruta_p256 = joinpath(ruta_carpeta, archivo_p256)
contenido_p256 = JSON3.read(read(ruta_p256, String))
datos_p256 = Int.(contenido_p256["reunion"])


archivo_p512 = "listas-posteo-con-perturbaciones-p=512.json"
ruta_p512 = joinpath(ruta_carpeta, archivo_p512)
contenido_p512 = JSON3.read(read(ruta_p512, String))
datos_p512 = Int.(contenido_p512["reunion"])

3080-element Vector{Int64}:
   260
   275
   294
   296
   314
 42610
   341
   384
   457
   529
 15293
   630
   675
     ⋮
  5929
 49018
 49550
 37111
 49570
  6185
 49716
 36199
 49918
 13772
 49975
 49987

In [63]:
#Evalución de los algoritmos por archivos
# p = 016
A = copy(datos_p016)
tiempo_heap_p016 = @elapsed begin
    _, comp_heap_p016 = heapsort_count!(A)
end

A = copy(datos_p016)
tiempo_merge_p016 = @elapsed begin
    _, comp_merge_p016 = mergesort_count(A)
end

A = copy(datos_p016)
tiempo_quick_p016 = @elapsed begin
    _, comp_quick_p016 = quicksort_mediana_count!(A)
end

A = copy(datos_p016)
tiempo_bubble_p016 = @elapsed begin
    _, comp_bubble_p016 = bubble_sort_adaptativo_count!(A)
end

# p = 032
A = copy(datos_p032)
tiempo_heap_p032 = @elapsed begin
    _, comp_heap_p032 = heapsort_count!(A)
end

A = copy(datos_p032)
tiempo_merge_p032 = @elapsed begin
    _, comp_merge_p032 = mergesort_count(A)
end

A = copy(datos_p032)
tiempo_quick_p032 = @elapsed begin
    _, comp_quick_p032 = quicksort_mediana_count!(A)
end

A = copy(datos_p032)
tiempo_bubble_p032 = @elapsed begin
    _, comp_bubble_p032 = bubble_sort_adaptativo_count!(A)
end

# p = 064
A = copy(datos_p064)
tiempo_heap_p064 = @elapsed begin
    _, comp_heap_p064 = heapsort_count!(A)
end

A = copy(datos_p064)
tiempo_merge_p064 = @elapsed begin
    _, comp_merge_p064 = mergesort_count(A)
end

A = copy(datos_p064)
tiempo_quick_p064 = @elapsed begin
    _, comp_quick_p064 = quicksort_mediana_count!(A)
end

A = copy(datos_p064)
tiempo_bubble_p064 = @elapsed begin
    _, comp_bubble_p064 = bubble_sort_adaptativo_count!(A)
end

# p = 128
A = copy(datos_p128)
tiempo_heap_p128 = @elapsed begin
    _, comp_heap_p128 = heapsort_count!(A)
end

A = copy(datos_p128)
tiempo_merge_p128 = @elapsed begin
    _, comp_merge_p128 = mergesort_count(A)
end

A = copy(datos_p128)
tiempo_quick_p128 = @elapsed begin
    _, comp_quick_p128 = quicksort_mediana_count!(A)
end

A = copy(datos_p128)
tiempo_bubble_p128 = @elapsed begin
    _, comp_bubble_p128 = bubble_sort_adaptativo_count!(A)
end

# p = 256
A = copy(datos_p256)
tiempo_heap_p256 = @elapsed begin
    _, comp_heap_p256 = heapsort_count!(A)
end

A = copy(datos_p256)
tiempo_merge_p256 = @elapsed begin
    _, comp_merge_p256 = mergesort_count(A)
end

A = copy(datos_p256)
tiempo_quick_p256 = @elapsed begin
    _, comp_quick_p256 = quicksort_mediana_count!(A)
end

A = copy(datos_p256)
tiempo_bubble_p256 = @elapsed begin
    _, comp_bubble_p256 = bubble_sort_adaptativo_count!(A)
end

# p = 512
A = copy(datos_p512)
tiempo_heap_p512 = @elapsed begin
    _, comp_heap_p512 = heapsort_count!(A)
end

A = copy(datos_p512)
tiempo_merge_p512 = @elapsed begin
    _, comp_merge_p512 = mergesort_count(A)
end

A = copy(datos_p512)
tiempo_quick_p512 = @elapsed begin
    _, comp_quick_p512 = quicksort_mediana_count!(A)
end

A = copy(datos_p512)
tiempo_bubble_p512 = @elapsed begin
    _, comp_bubble_p512 = bubble_sort_adaptativo_count!(A)
end

0.004026667

### Archivo p = 016

Para el archivo p = 016 se observa que los algoritmos con complejidad $(O(n \log n))$, como HeapSort, MergeSort y QuickSort, presentan un número de comparaciones considerablemente menor que Bubble sort adaptativo. En particular, MergeSort es el que realiza menos comparaciones, mientras que QuickSort presenta un número mayor debido al proceso de partición. HeapSort mantiene un comportamiento intermedio y relativamente estable. Por otro lado, Bubble sort adaptativo realiza muchas más comparaciones, lo cual era esperado debido a su complejidad $(O(n^2))$. En términos de tiempo de ejecución, QuickSort resulta ser el más rápido, mientras que Bubble sort presenta el mayor tiempo de ejecución.

In [66]:
using DataFrames

tabla_p016 = DataFrame(
    algoritmo = ["HeapSort", "MergeSort", "QuickSort mediana", "Bubble adaptativo"],
    comparaciones = [comp_heap_p016, comp_merge_p016, comp_quick_p016, comp_bubble_p016],
    tiempo = [tiempo_heap_p016, tiempo_merge_p016, tiempo_quick_p016, tiempo_bubble_p016]
)
print(tabla_p016)

4×3 DataFrame
 Row │ algoritmo          comparaciones  tiempo     
     │ String             Int64          Float64    
─────┼──────────────────────────────────────────────
   1 │ HeapSort                   64787  0.0203531
   2 │ MergeSort                  23867  0.031483
   3 │ QuickSort mediana         129215  0.0206103
   4 │ Bubble adaptativo        3705580  0.00957021

### Archivo p = 032

Para el archivo p = 032 se observa nuevamente que los algoritmos con complejidad $(O(n \log n))$, como HeapSort, MergeSort y QuickSort, presentan un número de comparaciones considerablemente menor que Bubble sort adaptativo. En este caso, MergeSort vuelve a ser el algoritmo con menor número de comparaciones, mientras que HeapSort mantiene un comportamiento estable y similar al observado anteriormente. QuickSort presenta un número mayor de comparaciones debido al proceso de partición, aunque sigue siendo menor que el de Bubble sort. Por otro lado, Bubble sort adaptativo continúa mostrando el mayor número de comparaciones, lo cual es consistente con su complejidad $(O(n^2))$. En términos de tiempo de ejecución, QuickSort sigue siendo el algoritmo más rápido, mientras que Bubble sort presenta nuevamente el mayor tiempo.

In [69]:
tabla_p032 = DataFrame(
    algoritmo = ["HeapSort", "MergeSort", "QuickSort mediana", "Bubble adaptativo"],
    comparaciones = [comp_heap_p032, comp_merge_p032, comp_quick_p032, comp_bubble_p032],
    tiempo = [tiempo_heap_p032, tiempo_merge_p032, tiempo_quick_p032, tiempo_bubble_p032]
)

print(tabla_p032)

4×3 DataFrame
 Row │ algoritmo          comparaciones  tiempo      
     │ String             Int64          Float64     
─────┼───────────────────────────────────────────────
   1 │ HeapSort                   64715  0.000213667
   2 │ MergeSort                  24791  0.000140791
   3 │ QuickSort mediana         104621  8.5875e-5
   4 │ Bubble adaptativo        4462282  0.00304521

### Archivo p = 064

Para el archivo p = 064 se observa un comportamiento similar al de los casos anteriores. MergeSort continúa siendo el algoritmo con el menor número de comparaciones, mientras que HeapSort mantiene valores relativamente estables en comparación con otros niveles de perturbación. QuickSort presenta un número de comparaciones mayor que MergeSort, pero destaca por tener el menor tiempo de ejecución. Por otro lado, Bubble sort adaptativo presenta un número de comparaciones considerablemente mayor que los demás algoritmos, lo cual también se refleja en su mayor tiempo de ejecución.

In [72]:
tabla_p064 = DataFrame(
    algoritmo = ["HeapSort", "MergeSort", "QuickSort mediana", "Bubble adaptativo"],
    comparaciones = [comp_heap_p064, comp_merge_p064, comp_quick_p064, comp_bubble_p064],
    tiempo = [tiempo_heap_p064, tiempo_merge_p064, tiempo_quick_p064, tiempo_bubble_p064]
)
print(tabla_p064)

4×3 DataFrame
 Row │ algoritmo          comparaciones  tiempo      
     │ String             Int64          Float64     
─────┼───────────────────────────────────────────────
   1 │ HeapSort                   64654  0.000228
   2 │ MergeSort                  26917  0.000162917
   3 │ QuickSort mediana          69104  8.8e-5
   4 │ Bubble adaptativo        4722157  0.00410304

### Archivo p = 128

En el caso del archivo p = 128 se mantiene la tendencia observada previamente. MergeSort nuevamente realiza el menor número de comparaciones, mientras que HeapSort presenta un comportamiento bastante estable. QuickSort muestra un número de comparaciones intermedio, pero continúa siendo uno de los algoritmos más rápidos en términos de tiempo de ejecución. En contraste, Bubble sort adaptativo vuelve a registrar el mayor número de comparaciones y el mayor tiempo, lo cual es consistente con la complejidad del algoritmo.

In [75]:
tabla_p128 = DataFrame(
    algoritmo = ["HeapSort", "MergeSort", "QuickSort mediana", "Bubble adaptativo"],
    comparaciones = [comp_heap_p128, comp_merge_p128, comp_quick_p128, comp_bubble_p128],
    tiempo = [tiempo_heap_p128, tiempo_merge_p128, tiempo_quick_p128, tiempo_bubble_p128]
)
print(tabla_p128)

4×3 DataFrame
 Row │ algoritmo          comparaciones  tiempo      
     │ String             Int64          Float64     
─────┼───────────────────────────────────────────────
   1 │ HeapSort                   64536  0.00023825
   2 │ MergeSort                  28882  0.000162084
   3 │ QuickSort mediana          68157  9.5084e-5
   4 │ Bubble adaptativo        4684707  0.0034275

### Archivo p = 256

Para el archivo p = 256 se observan resultados muy similares a los anteriores. MergeSort sigue siendo el algoritmo que realiza menos comparaciones, mientras que HeapSort mantiene valores cercanos a los observados en los otros archivos. QuickSort presenta una disminución en el número de comparaciones respecto a algunos casos previos y continúa mostrando uno de los menores tiempos de ejecución. En contraste, Bubble sort adaptativo mantiene el mayor número de comparaciones y el mayor tiempo de ejecución.

In [78]:
tabla_p256 = DataFrame(
    algoritmo = ["HeapSort", "MergeSort", "QuickSort mediana", "Bubble adaptativo"],
    comparaciones = [comp_heap_p256, comp_merge_p256, comp_quick_p256, comp_bubble_p256],
    tiempo = [tiempo_heap_p256, tiempo_merge_p256, tiempo_quick_p256, tiempo_bubble_p256]
)
print(tabla_p256)

4×3 DataFrame
 Row │ algoritmo          comparaciones  tiempo      
     │ String             Int64          Float64     
─────┼───────────────────────────────────────────────
   1 │ HeapSort                   64315  0.000220417
   2 │ MergeSort                  29803  0.000178292
   3 │ QuickSort mediana          50339  9.8125e-5
   4 │ Bubble adaptativo        4740799  0.00359967

### Archivo p = 512

Para el archivo p = 512 se mantiene el mismo patrón general en el comportamiento de los algoritmos. MergeSort continúa registrando el menor número de comparaciones, seguido por QuickSort y HeapSort. En términos de tiempo de ejecución, QuickSort mantiene uno de los tiempos más bajos, lo cual indica un buen desempeño práctico. HeapSort presenta un tiempo intermedio, mientras que Bubble sort adaptativo sigue mostrando el mayor número de comparaciones y el mayor tiempo de ejecución.

In [81]:
tabla_p512 = DataFrame(
    algoritmo = ["HeapSort", "MergeSort", "QuickSort mediana", "Bubble adaptativo"],
    comparaciones = [comp_heap_p512, comp_merge_p512, comp_quick_p512, comp_bubble_p512],
    tiempo = [tiempo_heap_p512, tiempo_merge_p512, tiempo_quick_p512, tiempo_bubble_p512]
)
print(tabla_p512)

4×3 DataFrame
 Row │ algoritmo          comparaciones  tiempo      
     │ String             Int64          Float64     
─────┼───────────────────────────────────────────────
   1 │ HeapSort                   63936  0.000219167
   2 │ MergeSort                  30845  0.0001865
   3 │ QuickSort mediana          41127  0.000102625
   4 │ Bubble adaptativo        4734400  0.00402667

In [83]:
tabla_general_comparaciones = DataFrame(
    archivo = String[],
    HeapSort = Int[],
    MergeSort = Int[],
    QuickSort_mediana = Int[],
    Bubble_adaptativo = Int[]
)

push!(tabla_general_comparaciones, ("p=016", comp_heap_p016, comp_merge_p016, comp_quick_p016, comp_bubble_p016))
push!(tabla_general_comparaciones, ("p=032", comp_heap_p032, comp_merge_p032, comp_quick_p032, comp_bubble_p032))
push!(tabla_general_comparaciones, ("p=064", comp_heap_p064, comp_merge_p064, comp_quick_p064, comp_bubble_p064))
push!(tabla_general_comparaciones, ("p=128", comp_heap_p128, comp_merge_p128, comp_quick_p128, comp_bubble_p128))
push!(tabla_general_comparaciones, ("p=256", comp_heap_p256, comp_merge_p256, comp_quick_p256, comp_bubble_p256))
push!(tabla_general_comparaciones, ("p=512", comp_heap_p512, comp_merge_p512, comp_quick_p512, comp_bubble_p512))

tabla_general_tiempos = DataFrame(
    archivo = String[],
    HeapSort = Float64[],
    MergeSort = Float64[],
    QuickSort_mediana = Float64[],
    Bubble_adaptativo = Float64[]
)

push!(tabla_general_tiempos, ("p=016", tiempo_heap_p016, tiempo_merge_p016, tiempo_quick_p016, tiempo_bubble_p016))
push!(tabla_general_tiempos, ("p=032", tiempo_heap_p032, tiempo_merge_p032, tiempo_quick_p032, tiempo_bubble_p032))
push!(tabla_general_tiempos, ("p=064", tiempo_heap_p064, tiempo_merge_p064, tiempo_quick_p064, tiempo_bubble_p064))
push!(tabla_general_tiempos, ("p=128", tiempo_heap_p128, tiempo_merge_p128, tiempo_quick_p128, tiempo_bubble_p128))
push!(tabla_general_tiempos, ("p=256", tiempo_heap_p256, tiempo_merge_p256, tiempo_quick_p256, tiempo_bubble_p256))
push!(tabla_general_tiempos, ("p=512", tiempo_heap_p512, tiempo_merge_p512, tiempo_quick_p512, tiempo_bubble_p512))

Row,archivo,HeapSort,MergeSort,QuickSort_mediana,Bubble_adaptativo
,String,Float64,Float64,Float64,Float64
1,p=016,0.0203531,0.031483,0.0206103,0.00957021
2,p=032,0.000213667,0.000140791,8.5875e-5,0.00304521
3,p=064,0.000228,0.000162917,8.8e-5,0.00410304
4,p=128,0.00023825,0.000162084,9.5084e-5,0.0034275
5,p=256,0.000220417,0.000178292,9.8125e-5,0.00359967
6,p=512,0.000219167,0.0001865,0.000102625,0.00402667


In [85]:
tabla_general_tiempos = DataFrame(
    archivo = String[],
    HeapSort = Float64[],
    MergeSort = Float64[],
    QuickSort_mediana = Float64[],
    Bubble_adaptativo = Float64[]
)

push!(tabla_general_tiempos, ("p=016", tiempo_heap_p016, tiempo_merge_p016, tiempo_quick_p016, tiempo_bubble_p016))
push!(tabla_general_tiempos, ("p=032", tiempo_heap_p032, tiempo_merge_p032, tiempo_quick_p032, tiempo_bubble_p032))
push!(tabla_general_tiempos, ("p=064", tiempo_heap_p064, tiempo_merge_p064, tiempo_quick_p064, tiempo_bubble_p064))
push!(tabla_general_tiempos, ("p=128", tiempo_heap_p128, tiempo_merge_p128, tiempo_quick_p128, tiempo_bubble_p128))
push!(tabla_general_tiempos, ("p=256", tiempo_heap_p256, tiempo_merge_p256, tiempo_quick_p256, tiempo_bubble_p256))
push!(tabla_general_tiempos, ("p=512", tiempo_heap_p512, tiempo_merge_p512, tiempo_quick_p512, tiempo_bubble_p512))

Row,archivo,HeapSort,MergeSort,QuickSort_mediana,Bubble_adaptativo
,String,Float64,Float64,Float64,Float64
1,p=016,0.0203531,0.031483,0.0206103,0.00957021
2,p=032,0.000213667,0.000140791,8.5875e-5,0.00304521
3,p=064,0.000228,0.000162917,8.8e-5,0.00410304
4,p=128,0.00023825,0.000162084,9.5084e-5,0.0034275
5,p=256,0.000220417,0.000178292,9.8125e-5,0.00359967
6,p=512,0.000219167,0.0001865,0.000102625,0.00402667


### Discusión de resultados

A partir de los resultados mostrados en la tablasy tiempo de ejecución, se pueden identificar patrones claros en el comportamiento de los algoritmos frente a los distintos niveles de perturbación de los datos. MergeSort es consistentemente el algoritmo que realiza el menor número de comparaciones en todos los archivos analizados, lo que indica que su desempeño es bastante estable y no depende significativamente del grado de desorden de los datos.

HeapSort también presenta un comportamiento relativamente constante conforme cambia el nivel de perturbación. Aunque realiza más comparaciones que MergeSort, sus valores se mantienen muy similares entre los distintos archivos, lo que sugiere que su desempeño tampoco se ve fuertemente afectado por el desorden inicial de los datos.

En el caso de QuickSort utilizando la mediana como pivote, se observa que el número de comparaciones tiende a disminuir conforme aumenta el valor de \(p\). Esto indica que el algoritmo puede beneficiarse cuando los datos presentan cierto orden o estructura, ya que la selección del pivote produce particiones más equilibradas. Además, QuickSort presenta generalmente los menores tiempos de ejecución.

Por otro lado, Bubble sort adaptativo muestra un comportamiento mucho más sensible al desorden de los datos. Aunque su versión adaptativa puede mejorar cuando los datos están casi ordenados, en los archivos analizados sigue presentando el mayor número de comparaciones y los mayores tiempos de ejecución. Esto se relaciona con su complejidad cuadrática $(O(n^2))$, que hace que su desempeño se degrade rápidamente conforme aumenta el tamaño del problema.

En general, los resultados muestran que los algoritmos con complejidad $(O(n \log n))$, como MergeSort, HeapSort y QuickSort, mantienen un desempeño eficiente y relativamente estable frente a distintos niveles de perturbación. En contraste, Bubble sort adaptativo resulta mucho menos eficiente para este tipo de conjuntos de datos.

## Conclusión

Los resultados obtenidos permiten comparar claramente el desempeño de los algoritmos de ordenamiento analizados. En general, los algoritmos con complejidad $(O(n \log n))$, como MergeSort, HeapSort y QuickSort, muestran un comportamiento mucho más eficiente tanto en número de comparaciones como en tiempo de ejecución. MergeSort mantiene el menor número de comparaciones en todos los archivos, mientras que QuickSort destaca por sus tiempos de ejecución bajos en la mayoría de los casos. HeapSort presenta un comportamiento bastante estable frente a los distintos niveles de perturbación. En contraste, Bubble sort adaptativo muestra valores mucho mayores tanto en comparaciones como en tiempo, lo que refleja las limitaciones de los algoritmos con complejidad cuadrática. 

En conjunto, los resultados muestran que los algoritmos con mejor complejidad, como MergeSort, HeapSort y QuickSort, son más adecuados para ordenar conjuntos de datos de este tamaño. Estos algoritmos mantienen un desempeño más estable tanto en el número de comparaciones como en el tiempo de ejecución, incluso cuando cambia el nivel de perturbación de los datos. En contraste, algoritmos más simples como Bubble sort adaptativo presentan un crecimiento mucho mayor en comparaciones y tiempo, lo que limita su eficiencia para problemas de mayor tamaño. Por esta razón, al trabajar con conjuntos de datos más grandes resulta más conveniente utilizar algoritmos con complejidad $(O(n \log n))$.

## Referencias

- Cormen, T. H., Leiserson, C. E., Rivest, R. L., & Stein, C. (2022). *Introduction to algorithms* (4th ed.). MIT Press.

- Estivill-Castro, V., & Wood, D. (1992). A survey of adaptive sorting algorithms. *ACM Computing Surveys*, 24(4), 441–476. 

- Knuth, D. E. (1998). *The art of computer programming, Volume 3: Sorting and searching* (2nd ed.). Addison-Wesley.